# NB4 · The Hands: tools, MCP, two harnesses, production grade

**Agentic Systems Engineering · Lecture 4 · Vizuara · Dr. Rajat Dandekar**

Northbridge Cycles builds e-bikes in Manchester and runs on **Ledgerline ERP 7.2** (1998). Its assistant, **Ask
Northbridge**, has a brain (Lecture 2) and a memory (Lecture 3). Today it gets hands: tools it can ask our code to run.
Emma Clarke, a purchase executive, asks two things on Monday 26 October 2026:

> **How many BAT-500 battery packs do we have in Manchester main stores?**
> **I need 2,000 of the 36 V 500 Wh battery packs from Voltway for the spring launch run.**

This notebook follows the lecture, section by section:

1. open a tool: `check_stock`'s `tool.json`, `tool.py` and the legacy services inside it
2. call it, and see every legacy envelope `{rc, err_cd, msg, data}` it turns into one clean result
3. the schemas: input and output, checked with a real JSON Schema validator and at the door
4. build a tool: `supplier_certificates` from its services, then your own `payment_terms`
5. MCP: start the server and talk JSON-RPC to it by hand: `initialize`, `tools/list`, `tools/call`
6. two harnesses: Claude Code's recorded run next to ours
7. our harness, with the context printed at every model call and real token counts
8. production checks: schemas, the error contract, latency and the idempotency fix

**No API key needed.** Every model call is replayed from our recorded runs (cassettes `L4v4_ours*`), so the notebook
shows exactly what happened. Five exercises are marked `TODO (exercise N)`, each with its solution behind
`SOLUTION = True`. Set it to `False` and try it yourself.

In [1]:
# Setup: install the Ledgerline sandbox and its recorded model replies. Runs in about a minute on Colab.
# REPLAY mode (the default when you have no key) plays back our recorded model calls, so every cell runs
# without an API key and shows the numbers from our own run. To call a model live, set a key below.
import os, sys, subprocess
COURSE_REPO = "https://github.com/RajatDandekar/agents-for-enterprises"   # course code (Ledgerline sandbox + reference agent)
try:
    import ledgerline  # already available (local checkout or Modal)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ledgerline-erp[all] @ git+{COURSE_REPO}.git#subdirectory=ledgerline"], check=True)
    import ledgerline
# os.environ["GEMINI_API_KEY"] = "..."        # optional: your own key, to run live
# os.environ["OPENROUTER_API_KEY"] = "..."    # optional: GPT, Claude, Llama ... via OpenRouter
os.environ.setdefault("LEDGER_LLM_MODE", "auto")  # auto = replay what was recorded, call live only for new requests
print("Ledgerline", ledgerline.__version__, "· model mode:", os.environ["LEDGER_LLM_MODE"])

Ledgerline 0.1.0 · model mode: replay


In [2]:
import inspect
import json
import pathlib
import statistics
import time

from ledgerline.erp.engine import ERP, SERVICES
from ledgerline.toolbox import ALL, REGISTRY, SPECS, Toolbox
from ledgerline.course.l4 import add_tool_folder, docs_root, handshake, lint_tool, service_definition, service_index, stdio_wire, validate_args
from ledgerline.course.l4.hands import CLAUDE_CODE, context_parts, hands_run, show_context


def pp(o, n=1400):
    """Print JSON, cut at n characters."""
    s = json.dumps(o, indent=1, ensure_ascii=False, default=str)
    print(s if len(s) <= n else s[:n] + " …")


print(len(ALL), "tool folders:", ", ".join(ALL))

16 tool folders: find_item, check_stock, find_supplier, check_budget, search_policy, create_purchase_request, list_my_approvals, approve_purchase_request, create_purchase_order, approve_purchase_order, release_purchase_order, order_status, receive_goods, register_supplier, send_message, supplier_certificates


## 1 · Open a tool

A tool is a function our code runs when the model asks for it. Each one is a folder with two files:
`tool.json`, what the model reads (its name, description and input schema), and `tool.py`, what runs.

In [3]:
spec = REGISTRY["check_stock"]
print(spec.folder.relative_to(spec.folder.parents[2]))
print((spec.folder / "tool.json").read_text())

ledgerline/toolbox/check_stock
{
  "name": "check_stock",
  "title": "Check stock",
  "category": "Look up",
  "description": "Stock of an item in every warehouse: on hand, allocated, on order, reorder level and days of cover.",
  "writes": false,
  "version": "1.0",
  "services": [
    "itm_get",
    "whs_list",
    "stk_query"
  ],
  "input_schema": {
    "type": "object",
    "properties": {
      "item_code": {
        "type": "string",
        "description": "Ledgerline item code, e.g. BAT-500"
      }
    },
    "required": [
      "item_code"
    ]
  },
  "output_schema": {
    "type": "object",
    "properties": {
      "ok": {
        "type": "boolean",
        "description": "true when it worked; false comes with error and next_step"
      },
      "item_code": {
        "type": "string",
        "description": ""
      },
      "warehouses": {
        "type": "array",
        "description": "per warehouse: on_hand, allocated, on_order, free"
      },
      "reorder_level": {

In [4]:
print(spec.source)

"""Check stock: Stock of an item in every warehouse: on hand, allocated, on order, reorder level and days of cover..

Input and output are described in tool.json next to this file."""
from ledgerline.agent.tools import ToolError, _ledgerline


def run(s, item_code):
    """Ask Ledgerline's stock enquiry for each warehouse and add it up."""
    item = _ledgerline(s, "itm_get", itm_cd=item_code)
    rows, reorder, daily = [], None, None
    for w in _ledgerline(s, "whs_list")["rows"]:
        try:
            r = _ledgerline(s, "stk_query", itm_cd=item_code, whs_cd=w["whs_cd"])
        except ToolError:
            continue  # no stock record in this warehouse
        reorder, daily = r["reorder_lvl"], r["avg_daily_issue"]
        rows.append({"warehouse": w["whs_cd"], "name": w["name"], "on_hand": r["on_hand"], "allocated": r["allocated"],
                     "on_order": r["on_order"], "free": r["on_hand"] - r["allocated"]})
    if not rows:
        raise ToolError(f"Ledgerline has no 

`tool.py` calls three of Ledgerline's own **services**: the legacy building blocks behind its screens. Where are they
written down? In Henrik Weber's documentation from Lecture 3, `northbridge-docs`: folder `04_service_definitions/` holds
one file per service. We clone it (the same repository as NB3), or use a copy you already have.

In [5]:
DOCS = docs_root(clone_to="northbridge-docs")      # $NORTHBRIDGE_DOCS, ./northbridge-docs, or git clone
SVC = DOCS / "04_service_definitions"
for d in sorted(x for x in SVC.iterdir() if x.is_dir()):
    fs = [f.name for f in d.iterdir()]
    print(f"{d.name + '/':20} {len(fs):3} files  ({sum(f.endswith('.yaml') for f in fs)} yaml, {sum(f.endswith('.xml') for f in fs)} xml)")
print(len(service_index()), "services in service_index.csv")

_legacy_6x/            3 files  (0 yaml, 3 xml)
accounts_payable/     71 files  (59 yaml, 12 xml)
admin/                61 files  (61 yaml, 0 xml)
inventory/            84 files  (84 yaml, 0 xml)
production/           67 files  (67 yaml, 0 xml)
purchase/             38 files  (38 yaml, 0 xml)
sales/                79 files  (70 yaml, 9 xml)
400 services in service_index.csv


In [6]:
print(service_definition("stk_query"))

# Ledgerline ERP 7.2 service definition, exported from the service registry on 26-10-2026 by JFLETCHER.
service: stk_query
title: STOCK STATUS ENQUIRY
module: PUR
kind: read
screen: INV-STK-02
since: "7.0"
endpoint: POST /ll72/svc/stk_query
auth: session token in header X-LL-Session; the user's role must allow the service
request:
  fields:
  - {name: itm_cd, type: str, required: true, desc: ITEM CD}
  - {name: whs_cd, type: str, required: false, desc: WHS CD}
response:
  envelope: {rc: "S | W | E", err_cd: "error code when rc is E", msg: "short message in capitals", data: "result"}
errors:
  - {code: ERR-STK-0001, msg: NO STOCK RECORD}
warnings: []
common_errors:
  - {code: ERR-SEC-0001, msg: ACCESS DENIED}
  - {code: ERR-SYS-0421, msg: "<FIELD> MANDATORY"}
  - {code: ERR-SYS-0422, msg: "UNKNOWN FIELD <FIELD> / BAD VALUE FOR <FIELD>"}
  - {code: ERR-SYS-0503, msg: "SERVER BUSY, TRY LATER"}
procedures: []
notes: "cover_days = (on_hand - allocated) / avg_daily_issue, rounded down."



The sandbox implements the same services in `ledgerline/erp/services.py`. The definitions and the code agree field
for field; here is the check for check_stock's three, and the code of `stk_query`.

In [7]:
import re

for name in spec.spec["services"]:
    sv = SERVICES[name]
    doc = re.findall(r"\{name: (\w+), type: \w+, required: (true|false)", service_definition(name))
    same = [(n, r == "true") for n, r in doc] == [(f.name, f.required) for f in sv.fields]
    print(f"{name:10} {sv.doc:22} kind={sv.kind:5} fields={[f.name for f in sv.fields]}  definition file matches: {same}")
print()
print(inspect.getsource(SERVICES["stk_query"].fn))

itm_get    ITEM MASTER GET        kind=read  fields=['itm_cd']  definition file matches: True
whs_list   WAREHOUSE LIST         kind=read  fields=[]  definition file matches: True
stk_query  STOCK STATUS ENQUIRY   kind=read  fields=['itm_cd', 'whs_cd']  definition file matches: True

@service("stk_query", "STOCK STATUS ENQUIRY", [F("itm_cd", req=True, desc="ITEM CD"), F("whs_cd", desc="WHS CD")],
         screen="INV-STK-02")
def stk_query(erp, user, itm_cd, whs_cd="WH-MAN-STR"):
    s = erp.q1("SELECT * FROM stock WHERE itm_cd=? AND whs_cd=?", itm_cd, whs_cd)
    i = erp.q1("SELECT avg_daily_issue, reorder_lvl FROM item WHERE itm_cd=?", itm_cd)
    if not s or not i:
        raise LedgerlineError("ERR-STK-0001", "NO STOCK RECORD")
    adi = i["avg_daily_issue"]
    return {**s, "avg_daily_issue": adi, "reorder_lvl": i["reorder_lvl"],
            "cover_days": int((s["on_hand"] - s["allocated"]) / adi) if adi else None}



## 2 · Call it

`Toolbox(erp, "ECLARKE")` binds the toolbox to Emma's own Ledgerline session: her permissions, nobody else's.
We call `check_stock` the way our code does when the model asks, and print every legacy call it made.

In [8]:
erp = ERP()
tb = Toolbox(erp, "ECLARKE")
n0 = len(erp.calls)
result = tb.call("check_stock", {"item_code": "BAT-500"})
for r in erp.calls[n0:]:
    print(f"{r.svc:10} {json.dumps(r.payload):46} -> rc={r.rc} err_cd={r.err_cd} data={json.dumps(r.data, default=str)[:110]}")
legacy = sum(len(json.dumps({"rc": r.rc, "err_cd": r.err_cd, "msg": r.msg, "data": r.data}, default=str)) for r in erp.calls[n0:])
print(f"\n{len(erp.calls) - n0} legacy calls, {legacy} bytes of legacy replies -> {len(json.dumps(result))} bytes of result:")
pp(result)

itm_get    {"itm_cd": "BAT-500"}                          -> rc=S err_cd=None data={"itm_cd": "BAT-500", "descr": "Battery pack, 36 V 500 Wh", "uom_cd": "NOS", "cell_safety": 1, "category": "SP
whs_list   {}                                             -> rc=S err_cd=None data={"rows": [{"whs_cd": "WH-MAN-STR", "name": "Manchester main stores", "ou_cd": "NBC-MAN-01"}, {"whs_cd": "WH-LD
stk_query  {"itm_cd": "BAT-500", "whs_cd": "WH-MAN-STR"}  -> rc=S err_cd=None data={"itm_cd": "BAT-500", "whs_cd": "WH-MAN-STR", "on_hand": 5400, "allocated": 0, "on_order": 0, "avg_daily_issue
stk_query  {"itm_cd": "BAT-500", "whs_cd": "WH-LDS-STR"}  -> rc=S err_cd=None data={"itm_cd": "BAT-500", "whs_cd": "WH-LDS-STR", "on_hand": 1200, "allocated": 0, "on_order": 0, "avg_daily_issue

4 legacy calls, 842 bytes of legacy replies -> 457 bytes of result:
{
 "ok": true,
 "item_code": "BAT-500",
 "description": "Battery pack, 36 V 500 Wh",
 "unit": "NOS",
 "warehouses": [
  {
   "warehouse": "WH-MAN-STR",
   

When something goes wrong, Ledgerline answers with a legacy code. The tool turns it into the **error contract**:
`ok=false`, `error` in words, `next_step`, and the legacy code kept as `ledgerline_code`. A wrong type never reaches
Ledgerline at all: it is refused at the door.

In [9]:
n1 = len(erp.calls)
print("BAT-5000 ->", tb.call("check_stock", {"item_code": "BAT-5000"}))
print("  legacy reply:", [(r.svc, r.rc, r.err_cd, r.msg) for r in erp.calls[n1:]])
n2 = len(erp.calls)
print("500 (an int) ->", tb.call("check_stock", {"item_code": 500}))
print("  Ledgerline calls:", len(erp.calls) - n2)

BAT-5000 -> {'ok': False, 'error': 'Ledgerline refused itm_get: Item not found (ERR-ITM-0010).', 'next_step': 'No item with that code. Use find_item with a description.', 'ledgerline_code': 'ERR-ITM-0010'}
  legacy reply: [('itm_get', 'E', 'ERR-ITM-0010', 'ITEM NOT FOUND')]
500 (an int) -> {'ok': False, 'error': "'item_code' should be string, got int.", 'next_step': 'Pass item_code as string.', 'ledgerline_code': ''}
  Ledgerline calls: 0


**TODO (exercise 1).** Check the stock of the hub motor `MTR-HUB-250`. From the `stk_query` replies alone, work out
how many days the free stock lasts (free = on hand − allocated, summed over warehouses, divided by the average daily
issue), and compare it with the tool's `cover_days`.

In [10]:
SOLUTION = True
if SOLUTION:
    n3 = len(erp.calls)
    motor = tb.call("check_stock", {"item_code": "MTR-HUB-250"})
    rows = [r.data for r in erp.calls[n3:] if r.svc == "stk_query" and r.rc == "S"]
    free = sum(d["on_hand"] - d["allocated"] for d in rows)
    daily = rows[0]["avg_daily_issue"] if rows else None
    print("by hand:", free, "free ÷", daily, "a day =", int(free / daily) if daily else None, "days")
    print("the tool says:", motor.get("cover_days"), "days ·", motor.get("free_total"), "free")

by hand: 48000 free ÷ 6000 a day = 8 days
the tool says: 8 days · 48000 free


## 3 · The schemas

The **input schema** is what the model must send; the **output schema** is what it gets back. Both are JSON Schema.
`validate_args` checks arguments with a real JSON Schema validator (`jsonschema`); `ToolSpec.check` is the toolbox's
own check at the door. Look at the last case: JSON Schema allows extra fields unless told otherwise; our door does not.

In [11]:
print(json.dumps(spec.spec["input_schema"], indent=1))
for args in [{"item_code": "BAT-500"}, {"item_code": 500}, {}, {"item_code": "BAT-500", "warehouse": "WH-MAN-STR"}]:
    print(f"\n{json.dumps(args):50} jsonschema: {validate_args('check_stock', args) or 'valid'}")
    try:
        print(" " * 50, "at the door:", spec.check(args))
    except Exception as e:  # noqa: BLE001
        print(" " * 50, "at the door: refused:", e)

{
 "type": "object",
 "properties": {
  "item_code": {
   "type": "string",
   "description": "Ledgerline item code, e.g. BAT-500"
  }
 },
 "required": [
  "item_code"
 ]
}



{"item_code": "BAT-500"}                           jsonschema: valid
                                                   at the door: {'item_code': 'BAT-500'}

{"item_code": 500}                                 jsonschema: ["item_code: 500 is not of type 'string'"]
                                                   at the door: refused: 'item_code' should be string, got int.

{}                                                 jsonschema: ["(top): 'item_code' is a required property"]
                                                   at the door: refused: Missing item_code.

{"item_code": "BAT-500", "warehouse": "WH-MAN-STR"} jsonschema: valid
                                                   at the door: refused: check_stock has no argument 'warehouse'.


In [12]:
import jsonschema

jsonschema.validate(result, spec.spec["output_schema"])
print("the real result is valid against check_stock's output schema")
print("fields in the result but not listed in the schema:", sorted(set(result) - set(spec.spec["output_schema"]["properties"])))

the real result is valid against check_stock's output schema
fields in the result but not listed in the schema: ['description', 'free_total', 'on_order_total', 'unit']


**TODO (exercise 2).** Write an input schema for a new tool `find_open_orders`: `supplier_code` (a string, required)
and `status` (one of `DRAFT`, `PENDING`, `RELEASED`, optional). Check one good and two bad argument sets with
`jsonschema`.

In [13]:
SOLUTION = True
if SOLUTION:
    schema = {"type": "object",
              "properties": {"supplier_code": {"type": "string", "description": "Ledgerline supplier code, e.g. VC-EIN-0042"},
                             "status": {"type": "string", "enum": ["DRAFT", "PENDING", "RELEASED"],
                                        "description": "optional: only orders in this status"}},
              "required": ["supplier_code"]}
    v = jsonschema.Draft202012Validator(schema)
    for a in [{"supplier_code": "VC-EIN-0042", "status": "RELEASED"}, {"status": "OPEN"}, {"supplier_code": 42}]:
        print(json.dumps(a), "->", [e.message for e in v.iter_errors(a)] or "valid")

{"supplier_code": "VC-EIN-0042", "status": "RELEASED"} -> valid
{"status": "OPEN"} -> ["'OPEN' is not one of ['DRAFT', 'PENDING', 'RELEASED']", "'supplier_code' is a required property"]
{"supplier_code": 42} -> ["42 is not of type 'string'"]


## 4 · Build a tool from scratch

`supplier_certificates` was built for this lecture in six steps: the job (Purchase Policy §6.2: lithium-cell parts
need a supplier with a valid UN 38.3 certificate), the services (`sup_get`, `sup_cert_get`, `itm_get`), the schema,
the code, the error contract and the tests. Here are its schema and code, then two real answers.

**Step 2, find the services.** Search the purchasing module's rows of `service_index.csv` for suppliers, certificates
and items, then read the definitions we chose.

In [14]:
pur = [r for r in service_index() if r["module"] == "PUR"]
cands = [r for r in pur if any(w in r["title"] for w in ("SUPPLIER", "ITEM", "CERT"))]
print(len(pur), "purchasing services;", len(cands), "candidates:")
for r in cands:
    print(f"   {r['service']:16} {r['title']:24} {r['kind']:5} {r['file']}")
for name in ("sup_get", "sup_cert_get"):
    print(f"\n--- 04_service_definitions/purchase/{name}.yaml"); print(service_definition(name))

38 purchasing services; 9 candidates:
   itm_get          ITEM MASTER GET          read  purchase/itm_get.yaml
   itm_search       ITEM ENQUIRY             read  purchase/itm_search.yaml
   po_hist          PO HISTORY BY ITEM       read  purchase/po_hist.yaml
   stk_reorder_list ITEMS BELOW REORDER LEVEL read  purchase/stk_reorder_list.yaml
   sup_cert_get     SUPPLIER CERT LIST       read  purchase/sup_cert_get.yaml
   sup_cert_upload  SUPPLIER CERT UPLOAD     write purchase/sup_cert_upload.yaml
   sup_create       SUPPLIER CREATE          write purchase/sup_create.yaml
   sup_get          SUPPLIER MASTER GET      read  purchase/sup_get.yaml
   sup_search       SUPPLIER ENQUIRY         read  purchase/sup_search.yaml

--- 04_service_definitions/purchase/sup_get.yaml
# Ledgerline ERP 7.2 service definition, exported from the service registry on 26-10-2026 by JFLETCHER.
service: sup_get
title: SUPPLIER MASTER GET
module: PUR
kind: read
screen: null
since: "7.0"
endpoint: POST /ll72/svc/s

In [15]:
sc = REGISTRY["supplier_certificates"]
print("services:", sc.spec["services"])
print(json.dumps(sc.spec["input_schema"], indent=1))
print(sc.source[sc.source.index("def run"):])

services: ['sup_get', 'sup_cert_get', 'itm_get']
{
 "type": "object",
 "properties": {
  "supplier_code": {
   "type": "string",
   "description": "Ledgerline supplier code, from find_supplier, e.g. NC-GDA-0063"
  },
  "item_code": {
   "type": "string",
   "description": "optional item code, e.g. BAT-500: also says whether this item needs UN 38.3 and whether the supplier qualifies"
  },
  "cert_type": {
   "type": "string",
   "description": "optional: only this certificate type, e.g. UN38.3"
  },
  "limit": {
   "type": "integer",
   "description": "optional: at most this many certificates per page (default 20, at most 50)"
  },
  "cursor": {
   "type": "string",
   "description": "optional: the next_cursor from the previous page"
  }
 },
 "required": [
  "supplier_code"
 ]
}
def run(s, supplier_code, item_code=None, cert_type=None, limit=None, cursor=None):
    """Read the supplier, its certificate list and (optionally) the item; say what is valid today."""
    size = PAGE if limit 

In [16]:
tb_all = Toolbox(ERP(), "ECLARKE", names=ALL)
pp(tb_all.call("supplier_certificates", {"supplier_code": "NC-GDA-0063", "item_code": "BAT-500"}))
pp(tb_all.call("supplier_certificates", {"supplier_code": "VC-EIN-0042", "item_code": "BAT-500"}))
for bad in [{"supplier_code": "XX-NOPE-0001"}, {}, {"supplier_code": 42}, {"supplier_code": "VC-EIN-0042", "limit": 500}]:
    print(json.dumps(bad), "->", tb_all.call("supplier_certificates", bad))

{
 "ok": true,
 "supplier_code": "NC-GDA-0063",
 "supplier_name": "Nordic Cell Co.",
 "supplier_status": "ACTIVE",
 "as_of": "2026-10-26",
 "certificates": [
  {
   "cert_type": "UN38.3",
   "doc_ref": "UN38.3-TR/NC/2025/090",
   "valid_to": "2026-09-30",
   "valid": false,
   "days_left": -26
  }
 ],
 "un38_3": "expired",
 "item": {
  "item_code": "BAT-500",
  "needs_un38_3": true,
  "supplier_qualifies": false
 },
 "total": 1,
 "next_cursor": null,
 "next_step": "BAT-500 contains lithium cells: Purchase Policy §6.2 needs a valid UN 38.3 certificate, and Nordic Cell Co. has an expired one. Do not order from them; use find_supplier for a certified supplier."
}
{
 "ok": true,
 "supplier_code": "VC-EIN-0042",
 "supplier_name": "Voltway Cells",
 "supplier_status": "ACTIVE",
 "as_of": "2026-10-26",
 "certificates": [
  {
   "cert_type": "UN38.3",
   "doc_ref": "UN38.3-TR/VC/2026/118",
   "valid_to": "2027-03-31",
   "valid": true,
   "days_left": 156
  }
 ],
 "un38_3": "valid",
 "item": {


**TODO (exercise 3).** Build your own tool, `payment_terms`: "What are Voltway's payment terms?" It needs two
services: `sup_get` (the supplier, with its `pt_cd`) and `pt_list` (every payment-terms code). Write its `tool.json`
and `tool.py` into `my_tools/payment_terms/`, load it with `add_tool_folder`, check it with `lint_tool` (the A4 checks)
and call it. This is the shape of A4, Part 1.

In [17]:
for n in ("sup_get", "pt_list"):     # the two definitions you need, from Henrik's documentation
    print(service_definition(n))

# Ledgerline ERP 7.2 service definition, exported from the service registry on 26-10-2026 by JFLETCHER.
service: sup_get
title: SUPPLIER MASTER GET
module: PUR
kind: read
screen: null
since: "7.0"
endpoint: POST /ll72/svc/sup_get
auth: session token in header X-LL-Session; the user's role must allow the service
request:
  fields:
  - {name: sup_cd, type: str, required: true, desc: SUP CD}
response:
  envelope: {rc: "S | W | E", err_cd: "error code when rc is E", msg: "short message in capitals", data: "result"}
errors:
  - {code: ERR-SUP-0030, msg: SUPPLIER NOT FOUND}
warnings: []
common_errors:
  - {code: ERR-SEC-0001, msg: ACCESS DENIED}
  - {code: ERR-SYS-0421, msg: "<FIELD> MANDATORY"}
  - {code: ERR-SYS-0422, msg: "UNKNOWN FIELD <FIELD> / BAD VALUE FOR <FIELD>"}
  - {code: ERR-SYS-0503, msg: "SERVER BUSY, TRY LATER"}
procedures: []

# Ledgerline ERP 7.2 service definition, exported from the service registry on 26-10-2026 by JFLETCHER.
service: pt_list
title: PAY TERM LIST
module: 

In [18]:
SOLUTION = True
if SOLUTION:
    folder = pathlib.Path("my_tools/payment_terms")
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "tool.json").write_text(json.dumps({
        "name": "payment_terms", "title": "Payment terms", "category": "Look up",
        "description": "A supplier's payment terms in Ledgerline: the terms code, what it means and the number of days to pay.",
        "writes": False, "version": "1.0", "services": ["sup_get", "pt_list"],
        "input_schema": {"type": "object", "properties": {"supplier_code": {"type": "string", "description": "Ledgerline supplier code, from find_supplier, e.g. VC-EIN-0042"}},
                         "required": ["supplier_code"]},
        "output_schema": {"type": "object", "properties": {"ok": {"type": "boolean"}, "supplier_code": {"type": "string"},
                                                           "supplier_name": {"type": "string"}, "terms_code": {"type": "string"},
                                                           "terms": {"type": "string"}, "days": {"type": "integer"}}, "required": ["ok"]},
        "example": {"supplier_code": "VC-EIN-0042"},
        "annotations": {"readOnlyHint": True, "destructiveHint": False, "idempotentHint": True, "openWorldHint": False}}, indent=2))
    (folder / "tool.py").write_text('''"""Payment terms: a supplier's payment terms, from two legacy reads."""
from ledgerline.agent.tools import ToolError, _ledgerline


def run(s, supplier_code):
    sup = _ledgerline(s, "sup_get", sup_cd=supplier_code.strip().upper())
    terms = {t["pt_cd"]: t for t in _ledgerline(s, "pt_list")["rows"]}
    t = terms.get(sup.get("pt_cd"))
    if not t:
        raise ToolError(f"{sup['sup_cd']} has no payment terms on file.", "Ask purchasing to set them in the supplier master.")
    return {"ok": True, "supplier_code": sup["sup_cd"], "supplier_name": sup["name"], "terms_code": t["pt_cd"],
            "terms": t["descr"], "days": t["days"]}
''')
    name = add_tool_folder("my_tools/payment_terms")     # your tool.json + tool.py, next to the shipped ones
    print("lint:", lint_tool(name) or "[] the A4 checks pass")
    print("jsonschema on a bad call:", validate_args(name, {"supplier_code": 42}))
    tb_mine = Toolbox(ERP(), "ECLARKE", names=[name])
    print(tb_mine.call(name, SPECS[name].spec["example"]))
    print(tb_mine.call(name, {"supplier_code": "XX-NOPE-0001"}))

lint: [] the A4 checks pass
jsonschema on a bad call: ["supplier_code: 42 is not of type 'string'"]
{'ok': True, 'supplier_code': 'VC-EIN-0042', 'supplier_name': 'Voltway Cells', 'terms_code': 'N45', 'terms': 'Net 45 days', 'days': 45}
{'ok': False, 'error': 'Ledgerline refused sup_get: Supplier not found (ERR-SUP-0030).', 'next_step': "No supplier with that code. Use find_supplier with the supplier's name.", 'ledgerline_code': 'ERR-SUP-0030'}


## 5 · MCP: the tool on a server, by hand

The Model Context Protocol (MCP) is the standard way for any app (Claude Code, Claude Desktop, our own harness) to
find and call tools. Our server is one command: `python -m ledgerline.toolbox.mcp --user ECLARKE --tools check_stock`.
Over **stdio** it reads one JSON-RPC message per line on stdin and answers on stdout. We start it the way Claude Code
does, as a child process, and type the messages ourselves.

In [19]:
wire = stdio_wire(handshake() + [
    {"jsonrpc": "2.0", "id": 2, "method": "tools/list"},
    {"jsonrpc": "2.0", "id": 3, "method": "tools/call", "params": {"name": "check_stock", "arguments": {"item_code": "BAT-500"}}},
    {"jsonrpc": "2.0", "id": 4, "method": "tools/call", "params": {"name": "check_stock", "arguments": {"item_code": "BAT-5000"}}},
], server_args=["--tools", "check_stock"])
for w in wire:
    print(w["dir"], json.dumps(w["msg"])[:230])

-> {"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {"protocolVersion": "2025-06-18", "capabilities": {}, "clientInfo": {"name": "nb4", "version": "1.0"}}}
<- {"jsonrpc": "2.0", "id": 1, "result": {"protocolVersion": "2025-06-18", "capabilities": {"logging": {}, "prompts": {"listChanged": true}, "resources": {"subscribe": false, "listChanged": true}, "tools": {"listChanged": true}}, "se
-> {"jsonrpc": "2.0", "method": "notifications/initialized"}
-> {"jsonrpc": "2.0", "id": 2, "method": "tools/list"}
<- {"jsonrpc": "2.0", "id": 2, "result": {"tools": [{"_meta": {"ledgerline/version": "1.0", "ledgerline/category": "Look up", "ledgerline/writes": false, "ledgerline/services": ["itm_get", "whs_list", "stk_query"], "ledgerline/policy
-> {"jsonrpc": "2.0", "id": 3, "method": "tools/call", "params": {"name": "check_stock", "arguments": {"item_code": "BAT-500"}}}
<- {"jsonrpc": "2.0", "id": 3, "result": {"content": [{"text": "{\"ok\": true, \"item_code\": \"BAT-500\", \"descripti

In [20]:
by_id = {w["msg"].get("id"): w["msg"] for w in wire if w["dir"] == "<-"}
tool = by_id[2]["result"]["tools"][0]
print("protocol agreed:", by_id[1]["result"]["protocolVersion"], "· server:", by_id[1]["result"]["serverInfo"])
print("definition fields:", sorted(tool))
print("annotations:", tool.get("annotations"))
print("_meta:", {k: v for k, v in tool.get("_meta", {}).items() if k.startswith("ledgerline/") and k != "ledgerline/policy"})
for i in (3, 4):
    res = by_id[i]["result"]
    print(f"\ntools/call id {i}: isError={res.get('isError', False)} · result fields {sorted(res)}")
    print("   structuredContent:", json.dumps(res.get("structuredContent"))[:200])

protocol agreed: 2025-06-18 · server: {'name': 'ledgerline-toolbox', 'version': '4.1.6'}
definition fields: ['_meta', 'annotations', 'description', 'inputSchema', 'name', 'outputSchema', 'title']
annotations: {'destructiveHint': False, 'idempotentHint': True, 'openWorldHint': False, 'readOnlyHint': True, 'title': 'Check stock'}
_meta: {'ledgerline/version': '1.0', 'ledgerline/category': 'Look up', 'ledgerline/writes': False, 'ledgerline/services': ['itm_get', 'whs_list', 'stk_query'], 'ledgerline/release_notes': None}

tools/call id 3: isError=False · result fields ['content', 'isError', 'structuredContent']
   structuredContent: {"ok": true, "item_code": "BAT-500", "description": "Battery pack, 36 V 500 Wh", "unit": "NOS", "warehouses": [{"warehouse": "WH-MAN-STR", "name": "Manchester main stores", "on_hand": 5400, "allocated

tools/call id 4: isError=True · result fields ['content', 'isError', 'structuredContent']
   structuredContent: {"ok": false, "error": "Ledgerline refused itm_g

**TODO (exercise 4).** Start the server with every tool (`server_args=["--all"]`), count the tools in `tools/list`,
and call `supplier_certificates` for Nordic Cell Co. (`NC-GDA-0063`) with `BAT-500`. Is it an error (`isError`), or a
business answer?

In [21]:
SOLUTION = True
if SOLUTION:
    w2 = stdio_wire(handshake() + [
        {"jsonrpc": "2.0", "id": 2, "method": "tools/list"},
        {"jsonrpc": "2.0", "id": 3, "method": "tools/call", "params": {"name": "supplier_certificates",
                                                                     "arguments": {"supplier_code": "NC-GDA-0063", "item_code": "BAT-500"}}}],
        server_args=["--all"])
    m = {w["msg"].get("id"): w["msg"] for w in w2 if w["dir"] == "<-"}
    print(len(m[2]["result"]["tools"]), "tools")
    r3 = m[3]["result"]
    print("isError:", r3.get("isError", False), "· un38_3:", r3["structuredContent"]["un38_3"], "·", r3["structuredContent"].get("next_step"))

16 tools
isError: False · un38_3: expired · BAT-500 contains lithium cells: Purchase Policy §6.2 needs a valid UN 38.3 certificate, and Nordic Cell Co. has an expired one. Do not order from them; use find_supplier for a certified supplier.


To connect the same server to **Claude Code**, run in a terminal (with `ledgerline-erp[mcp]` installed):

```
claude mcp add northbridge -- python -m ledgerline.toolbox.mcp --user ECLARKE --tools check_stock
```

and for **Claude Desktop**, add it to `claude_desktop_config.json`:

```
{"mcpServers": {"northbridge": {"command": "/full/path/to/python",
  "args": ["-m", "ledgerline.toolbox.mcp", "--user", "ECLARKE", "--tools", "check_stock"]}}}
```

## 6 · Two harnesses

The **harness** is everything around the model: the system prompt, which tools it gets, the loop that runs its calls,
the guards, the limits, the logs. Claude Code is Anthropic's harness; Ask Northbridge's loop
(`ledgerline/experience/harness.py`) is ours. Here is Claude Code's recorded run (claude-sonnet-5.5, headless, on our
MCP server with two tools) next to ours on the same question.

In [22]:
cc = CLAUDE_CODE["single"]
for t in cc["turns"]:
    print("EMMA:", t["user"])
    for s in t["steps"]:
        print("  ", s["kind"], s.get("tool") or "", json.dumps(s.get("args")) if s.get("args") else (s.get("text") or "")[:150])
    print("   Claude Code turns:", t["num_turns"], "· tokens:", t["usage"])

EMMA: How many BAT-500 battery packs do we have in Manchester main stores?
   call check_stock {"item_code": "BAT-500"}
   result check_stock {"ok":true,"item_code":"BAT-500","description":"Battery pack, 36 V 500 Wh","unit":"NOS","warehouses":[{"warehouse":"WH-MAN-STR","name":"Manchester mai
   text  Manchester main stores (WH-MAN-STR) has 5,400 BAT-500 battery packs on hand. None are allocated, so all 5,400 are free.
   Claude Code turns: 2 · tokens: {'input_tokens': 180, 'cache_creation_input_tokens': 5651, 'cache_read_input_tokens': 5236, 'output_tokens': 119}


In [23]:
single = hands_run("single")
for t in single["turns"]:
    print("EMMA:", t["user"])
    for c in t["cards"]:
        print("   call", c["tool"], json.dumps(c["args"]), "->", json.dumps(c["result"])[:150])
    print("   REPLY:", t["reply"])
    print("   our harness: model calls", t["model_calls"], "· tokens in", t["tokens_in"], "· out", t["tokens_out"])

EMMA: How many BAT-500 battery packs do we have in Manchester main stores?
   call check_stock {"item_code": "BAT-500"} -> {"ok": true, "item_code": "BAT-500", "description": "Battery pack, 36 V 500 Wh", "unit": "NOS", "warehouses": [{"warehouse": "WH-MAN-STR", "name": "Ma
   REPLY: Manchester main stores currently has 5,400 BAT-500 battery packs on hand (all free, none allocated or on order). 

Across all locations, there are 6,600 units in total, giving 11 days of stock cover.
   our harness: model calls 2 · tokens in 1850 · out 143


## 7 · What the model sees

Our harness kept a copy of the messages at every model call, and the provider's own token count. `show_context`
prints the context part by part. The parts add up exactly to each call's `prompt_tokens`. Nothing else ever reaches
the model: not `tool.py`, not the services or their envelopes, not the database, not Emma's session.

In [24]:
show_context(single)

-- model call 1 of 2: prompt_tokens 799 --
   +    654  system prompt                    
   +    128  tool definitions (2)             
   +     17  user message                     How many BAT-500 battery packs do we have in Manch
     = 799 tokens in the context (matches prompt_tokens)
     model wrote (86 tokens): check_stock({"item_code": "BAT-500"})
-- model call 2 of 2: prompt_tokens 1,051 --
   +     23  tool call                        [{"check_stock": {"item_code": "BAT-500"}}]
   +    229  tool result                      {"ok": true, "item_code": "BAT-500", "description"
     = 1,051 tokens in the context (matches prompt_tokens)
     model wrote (57 tokens): 'Manchester main stores currently has 5,400 BAT-500 battery packs on hand (all free, none a'
Never in the context: tool.py, the Ledgerline services and their {rc, err_cd, msg, data} replies, the database, Emma's session and password, the MCP plumbing.


Now Emma's whole order, on Lecture 4's "hands" prompt (the rules, no Northbridge codes): 16 tools, three turns from
Emma. Watch the context grow, and notice that the 16 tool definitions are re-sent with every call.

In [25]:
order = hands_run("order_hands")
parts = context_parts(order)
show_context(order, parts, width=80)

-- model call 1 of 11: prompt_tokens 1,916 --
   +    331  system prompt                    
   +  1,555  tool definitions (16)            
   +     30  user message                     I need 2,000 of the 36 V 500 Wh battery 
     = 1,916 tokens in the context (matches prompt_tokens)
     model wrote (153 tokens): find_item({"query": "36 V 500 Wh battery pack"}), find_supplier({"name": "Voltway"}), check_budget({})
-- model call 2 of 11: prompt_tokens 2,226 --
   +     59  tool call                        [{"find_item": {"query": "36 V 500 Wh ba
   +     57  tool result                      {"ok": true, "items": [{"item_code": "BA
   +    147  tool result                      {"ok": true, "matches": [{"supplier_code
   +     47  tool result                      {"ok": true, "open": [{"code": "PRD-07",
     = 2,226 tokens in the context (matches prompt_tokens)
     model wrote (45 tokens): check_budget({"cost_centre": "PRD-07"}), search_policy({"question": "purchase request vs purchase

In [26]:
seq_ours = [c["tool"] for t in order["turns"] for c in t["cards"]]
seq_cc = [s["tool"] for t in CLAUDE_CODE["order_hands"]["turns"] for s in t["steps"] if s["kind"] == "call"]
print(f"our harness ({len(seq_ours)} calls):", " -> ".join(seq_ours))
print(f"Claude Code ({len(seq_cc)} calls):", " -> ".join(seq_cc))
print("final reply (ours):", order["turns"][-1]["reply"])
tools_tok = next(p["tokens"] for p in parts if p["part"].startswith("tool definitions"))
calls = [s for s in order["steps"] if s["tools"] is not None]
print(f"{len(calls)} model calls read {sum(s['prompt_tokens'] for s in calls):,} tokens; the tool definitions alone: "
      f"{len(calls)} x {tools_tok:,} = {len(calls) * tools_tok:,}")

our harness (11 calls): find_item -> find_supplier -> check_budget -> check_budget -> search_policy -> search_policy -> check_stock -> create_purchase_request -> supplier_certificates -> create_purchase_request -> create_purchase_request
Claude Code (8 calls): find_item -> find_supplier -> supplier_certificates -> check_stock -> check_budget -> create_purchase_request -> check_budget -> create_purchase_request
final reply (ours): Purchase request PR/2026/04417 has been submitted. Its status is WAITING FOR APPROVAL, and it is currently waiting for Daniel Okafor's approval. Nothing has been ordered yet; once Daniel approves it, we can create the purchase order.
11 model calls read 34,984 tokens; the tool definitions alone: 11 x 1,555 = 17,105


## 8 · Production checks

A demo works once. Production is measured. Four of the lecture's ten criteria, on the whole toolbox, in this sandbox.

**Schema.** Every tool's own example, called on a fresh Ledgerline; every ok result checked against its output schema.

In [27]:
tb_p = Toolbox(ERP(), "ECLARKE", names=ALL)
viol = {}
for n in ALL:
    out = tb_p.call(n, REGISTRY[n].spec.get("example") or {})
    if out.get("ok"):
        errs = [e.message for e in jsonschema.Draft202012Validator(REGISTRY[n].spec["output_schema"]).iter_errors(out)]
        viol[n] = errs
    print(f"{n:26} ok={out.get('ok')!s:5} {'schema violations: ' + str(viol[n]) if viol.get(n) else ''}{'' if out.get('ok') else out.get('error', '')[:70]}")
print(f"\n{sum(1 for v in viol.values() if not v)} of {len(viol)} ok results valid against their output schemas")

find_item                  ok=True  
check_stock                ok=True  
find_supplier              ok=True  
check_budget               ok=True  


/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files:  20%|██        | 1/5 [00:00<00:00,  7.06it/s]

Fetching 5 files: 100%|██████████| 5/5 [00:01<00:00,  2.88it/s]

Fetching 5 files: 100%|██████████| 5/5 [00:01<00:00,  2.99it/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files:  20%|██        | 1/5 [00:00<00:00,  6.95it/s]

Fetching 5 files: 100%|██████████| 5/5 [00:01<00:00,  3.45it/s]

Fetching 5 files: 100%|██████████| 5/5 [00:01<00:00,  3.55it/s]

search_policy              ok=True  
create_purchase_request    ok=True  
list_my_approvals          ok=True  
approve_purchase_request   ok=False Ledgerline refused pr_approve: Access denied (ERR-SEC-0001).
create_purchase_order      ok=False Ledgerline refused pr_get: Pr not found (ERR-PR-0001).
approve_purchase_order     ok=False Ledgerline refused po_approve: Access denied (ERR-SEC-0001).
release_purchase_order     ok=False Ledgerline refused po_get: Po not found (ERR-PO-0001).
order_status               ok=False Ledgerline refused po_get: Po not found (ERR-PO-0001).
receive_goods              ok=False Ledgerline refused po_get: Po not found (ERR-PO-0001).
register_supplier          ok=False Missing name, city, state, vat_no.
send_message               ok=True  
supplier_certificates      ok=True  

9 of 9 ok results valid against their output schemas


**Reliability.** Bad arguments of every kind, on every tool. Every answer must be inside the error contract
(`ok`, and when false: `error` and `next_step`), and none may be a non-business failure (`INTERNAL`, or a bad value
that reached Ledgerline: `ERR-SYS-0422`).

In [28]:
BAD = [None, "", " ", -1, 0, 10**9, "x" * 200, [], {}]
calls = bad_contract = non_business = 0
for n in ALL:
    props = REGISTRY[n].spec["input_schema"].get("properties", {})
    for k in props:
        for v in BAD:
            args = {**(REGISTRY[n].spec.get("example") or {}), k: v, "confirm": False} if "confirm" in props else {**(REGISTRY[n].spec.get("example") or {}), k: v}
            out = tb_p.call(n, args)
            calls += 1
            if not isinstance(out, dict) or "ok" not in out or (out["ok"] is False and not (out.get("error") and out.get("next_step"))):
                bad_contract += 1
            if out.get("ledgerline_code") in ("INTERNAL", "ERR-SYS-0422"):
                non_business += 1
print(f"{calls} fuzzed calls · outside the error contract: {bad_contract} · non-business failures: {non_business}")

396 fuzzed calls · outside the error contract: 0 · non-business failures: 0


**Latency.** The tool's own time, in this sandbox (Ledgerline here is an in-memory database, so this is mostly our
code). The lecture's threshold for the overhead is p50 ≤ 50 ms, p95 ≤ 100 ms, p99 ≤ 200 ms.

In [29]:
ms = []
for _ in range(300):
    t0 = time.perf_counter()
    tb_p.call("check_stock", {"item_code": "BAT-500"})
    ms.append((time.perf_counter() - t0) * 1000)
q = statistics.quantiles(ms, n=100)
print(f"check_stock, 300 calls: p50 {q[49]:.2f} ms · p95 {q[94]:.2f} ms · p99 {q[98]:.2f} ms")

check_stock, 300 calls: p50 0.28 ms · p95 0.32 ms · p99 0.39 ms


**Idempotency: the fix from the lecture.** A write is retried with the date written differently: "9 Nov 2026", then
"2026-11-09". Before the fix, the idempotency key was built from the arguments as written, so the two calls made two
purchase requests. Now the door rewrites `need_by` as YYYY-MM-DD before anything uses it.

**TODO (exercise 5).** Before you run the next cell, predict: how many purchase requests will Ledgerline hold, and
what will the second reply say? Then show why the old key failed, with `content_key` on the two dates as written.

In [30]:
from ledgerline.agent.tools import content_key

erp_i = ERP()
tb_i = Toolbox(erp_i, "ECLARKE", names=ALL)
first = {"item_code": "BAT-500", "quantity": 2000, "cost_centre": "PRD-07", "need_by": "9 Nov 2026", "confirm": True}
retry = {**first, "need_by": "2026-11-09"}
a = tb_i.call("create_purchase_request", first)
b = tb_i.call("create_purchase_request", retry)
print(a["request_no"], b.get("request_no"), "already_done:", b.get("already_done"))
print("purchase requests in Ledgerline today:", erp_i.q("SELECT pr_no, need_dt FROM pr_hdr WHERE raised_on >= '2026-10-26'"))
SOLUTION = True
if SOLUTION:
    print("key, dates as written:     ", content_key("BAT-500", 2000, "PRD-07", "9 Nov 2026"), content_key("BAT-500", 2000, "PRD-07", "2026-11-09"))
    clean = REGISTRY["create_purchase_request"].check(first)["need_by"]
    print("key, after the door:       ", content_key("BAT-500", 2000, "PRD-07", clean), content_key("BAT-500", 2000, "PRD-07", "2026-11-09"))

PR/2026/04417 PR/2026/04417 already_done: True
purchase requests in Ledgerline today: [{'pr_no': 'PR/2026/04417', 'need_dt': '2026-11-09'}]
key, dates as written:      K3F864F714B33 K45F765BDEEAB
key, after the door:        K45F765BDEEAB K45F765BDEEAB


## Your turn: A4

A4 asks you to do all of this for **your own domain**: build one tool from your system's service definitions (Part 1,
exercise 3 is the template), put it on MCP and paste the raw JSON-RPC (Part 2, section 5), run the same task on Claude
Code and on this notebook's harness (Part 3, sections 6–7), and measure it against at least five of the ten production
criteria, fixing one (Part 4, section 8). Details in `A4.md`.